# Cats vs Dogs — Training Notebook

This notebook trains the image classifier used by every client of the project (mobile, web, desktop).

**Plan**
1. Download the Kaggle *Cats vs Dogs* dataset (Microsoft mirror, ~25k images) and remove corrupted files.
2. Split into **train (80%) / validation (10%) / test (10%)**.
3. Train a **baseline CNN from scratch**, the same approach we saw in class.
4. Train a **MobileNetV2 transfer-learning model** (pretrained on ImageNet), then fine-tune it.
5. Compare both on the held-out test set and export the best model for the Flask backend.

> **Runtime:** in Colab use `Runtime → Change runtime type → T4 GPU` before running.

In [ ]:
import json
import pathlib
import shutil
import time
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (training will be slow!)")

## 1. Configuration

In [ ]:
IMG_SIZE = 160          # MobileNetV2 has ImageNet weights for 160x160
BATCH_SIZE = 64
SEED = 1337

BASELINE_EPOCHS = 15    # early stopping may end it sooner
HEAD_EPOCHS = 5         # transfer learning: train only the new classifier head
FINE_TUNE_EPOCHS = 5    # transfer learning: unfreeze the top of MobileNetV2
FINE_TUNE_AT = 100      # MobileNetV2 has 154 layers; layers from this index on get unfrozen

CLASS_NAMES = ["Cat", "Dog"]  # label 0 = Cat, label 1 = Dog

DATA_DIR = pathlib.Path("PetImages")
OUT_DIR = pathlib.Path("export")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

keras.utils.set_random_seed(SEED)

## 2. Download the dataset

Same dataset as the Keras example from class (Kaggle *Dogs vs Cats*, hosted by Microsoft).

In [ ]:
URL = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"

if not DATA_DIR.exists():
    zip_path = "kagglecatsanddogs_5340.zip"
    print("Downloading...")
    urllib.request.urlretrieve(URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(".")
    print("Done")

for cls in CLASS_NAMES:
    print(cls, len(list((DATA_DIR / cls).iterdir())), "files")

## 3. Remove corrupted images

Some files in this dataset are corrupted or are not real JPEGs (they have no `JFIF` header). TensorFlow can't decode them, so we delete them.

In [ ]:
removed = 0
for cls in CLASS_NAMES:
    for path in (DATA_DIR / cls).iterdir():
        with open(path, "rb") as f:
            is_jfif = b"JFIF" in f.peek(10)
        if not is_jfif:
            path.unlink()
            removed += 1

print(f"Removed {removed} corrupted / non-JPEG files")
for cls in CLASS_NAMES:
    print(cls, len(list((DATA_DIR / cls).glob("*.jpg"))), "images")

## 4. Train / validation / test split

We build the split ourselves (stratified, fixed seed) so the **test set is never seen during training** and stays identical between the two models.

In [ ]:
paths, labels = [], []
for label, cls in enumerate(CLASS_NAMES):
    for p in sorted((DATA_DIR / cls).glob("*.jpg")):
        paths.append(str(p))
        labels.append(label)
paths, labels = np.array(paths), np.array(labels)

train_p, temp_p, train_y, temp_y = train_test_split(
    paths, labels, test_size=0.2, stratify=labels, random_state=SEED)
val_p, test_p, val_y, test_y = train_test_split(
    temp_p, temp_y, test_size=0.5, stratify=temp_y, random_state=SEED)

print(f"train: {len(train_p)}  val: {len(val_p)}  test: {len(test_p)}")


def load_image(path, label):
    # Must match the backend preprocessing: RGB -> bilinear resize -> float32 in [0, 255].
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img, tf.cast(label, tf.float32)


def make_dataset(p, y, training):
    ds = tf.data.Dataset.from_tensor_slices((p, y))
    if training:
        ds = ds.shuffle(len(p), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).ignore_errors()
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(train_p, train_y, training=True)
val_ds = make_dataset(val_p, val_y, training=False)
test_ds = make_dataset(test_p, test_y, training=False)

In [ ]:
images, batch_labels = next(iter(train_ds))
plt.figure(figsize=(10, 10))
for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(CLASS_NAMES[int(batch_labels[i])])
    plt.axis("off")
plt.suptitle("Training samples")
plt.savefig(FIG_DIR / "samples.png", bbox_inches="tight")
plt.show()

## 5. Data augmentation

Random flips, rotations and zooms create variations of the training images so the model generalizes better.
These layers are **only active during training**; at prediction time they do nothing.

In [ ]:
def make_augmentation():
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ], name="augmentation")


aug = make_augmentation()
plt.figure(figsize=(10, 10))
for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(aug(images[:1], training=True)[0].numpy().astype("uint8"))
    plt.axis("off")
plt.suptitle("Same image, different augmentations")
plt.savefig(FIG_DIR / "augmentation.png", bbox_inches="tight")
plt.show()

## 6. Model A — Baseline CNN from scratch

A small convolutional network trained from random weights (like the class example).

In [ ]:
def build_baseline():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = make_augmentation()(inputs)
    x = layers.Rescaling(1.0 / 255)(x)
    for filters in (32, 64, 128, 256):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")


baseline = build_baseline()
baseline.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss="binary_crossentropy", metrics=["accuracy"])
baseline.summary()

In [ ]:
start = time.time()
baseline_history = baseline.fit(
    train_ds, validation_data=val_ds, epochs=BASELINE_EPOCHS,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])
baseline_time = time.time() - start
print(f"Baseline training time: {baseline_time / 60:.1f} min")

## 7. Model B — Transfer learning with MobileNetV2

MobileNetV2 was already trained on 1.4M ImageNet images, so it already "knows" edges, textures, fur, eyes...
We reuse it as a feature extractor and only train a new classifier on top.

- **Phase 1:** freeze MobileNetV2, train only the new head.
- **Phase 2 (fine-tuning):** unfreeze the top layers of MobileNetV2 and train with a very low learning rate.

The preprocessing (`Rescaling` to [-1, 1]) is **inside the model**, so the backend only needs to resize the image.

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base_model.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = make_augmentation()(inputs)
x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)   # same as mobilenet_v2.preprocess_input
x = base_model(x, training=False)                  # keep BatchNorm layers in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs, name="mobilenetv2_cats_dogs")

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
start = time.time()
head_history = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False
print("Trainable layers in MobileNetV2:", sum(l.trainable for l in base_model.layers))

# Recompile with a much lower learning rate so we don't destroy the pretrained weights.
model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="binary_crossentropy", metrics=["accuracy"])
fine_history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=HEAD_EPOCHS + FINE_TUNE_EPOCHS, initial_epoch=len(head_history.epoch))
transfer_time = time.time() - start
print(f"Transfer learning training time: {transfer_time / 60:.1f} min")

## 8. Training curves

In [ ]:
def merge(*histories):
    out = {}
    for h in histories:
        for k, v in h.history.items():
            out.setdefault(k, []).extend(v)
    return out


curves = {
    "Baseline CNN": merge(baseline_history),
    "MobileNetV2": merge(head_history, fine_history),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for col, (name, h) in enumerate(curves.items()):
    for row, metric in enumerate(["accuracy", "loss"]):
        ax = axes[row, col]
        ax.plot(h[metric], label="train")
        ax.plot(h["val_" + metric], label="validation")
        if name == "MobileNetV2":
            ax.axvline(HEAD_EPOCHS - 1, color="gray", linestyle="--", label="start fine-tuning")
        ax.set_title(f"{name} — {metric}")
        ax.set_xlabel("epoch")
        ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "training_curves.png", bbox_inches="tight")
plt.show()

## 9. Evaluation on the held-out test set

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds]).astype(int)

results = {}
for name, m in [("Baseline CNN", baseline), ("MobileNetV2", model)]:
    y_prob = m.predict(test_ds, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    acc = float((y_pred == y_true).mean())
    results[name] = {"y_prob": y_prob, "y_pred": y_pred, "accuracy": acc}
    print(f"===== {name}: test accuracy = {acc:.4f} =====")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, r) in zip(axes, results.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_true, r["y_pred"]),
                           display_labels=CLASS_NAMES).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name} (acc {r['accuracy']:.2%})")
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrices.png", bbox_inches="tight")
plt.show()

### Choosing the "uncertain" threshold

The model only knows two classes, so for a photo of a car it still has to answer "cat" or "dog".
The backend will answer **"uncertain"** when the confidence is below a threshold. Here we check how much of the
(real cat/dog) test set would be flagged at different thresholds, to pick a sensible value.

In [ ]:
probs = results["MobileNetV2"]["y_prob"]
confidence = np.maximum(probs, 1 - probs)
correct = results["MobileNetV2"]["y_pred"] == y_true

print("threshold | flagged uncertain | accuracy on the rest")
for t in [0.6, 0.7, 0.8, 0.9, 0.95]:
    keep = confidence >= t
    print(f"   {t:.2f}   |      {1 - keep.mean():6.2%}      |     {correct[keep].mean():.4f}")

plt.hist(confidence[correct], bins=30, alpha=0.7, label="correct")
plt.hist(confidence[~correct], bins=30, alpha=0.7, label="wrong")
plt.yscale("log")
plt.xlabel("confidence")
plt.ylabel("images (log scale)")
plt.title("MobileNetV2 confidence on the test set")
plt.legend()
plt.savefig(FIG_DIR / "confidence_histogram.png", bbox_inches="tight")
plt.show()

In [ ]:
images, batch_labels = next(iter(test_ds))
batch_probs = model.predict(images, verbose=0).ravel()

plt.figure(figsize=(14, 10))
for i in range(12):
    ax = plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    p = batch_probs[i]
    pred = CLASS_NAMES[int(p >= 0.5)]
    conf = max(p, 1 - p)
    ok = pred == CLASS_NAMES[int(batch_labels[i])]
    plt.title(f"{pred} ({conf:.1%})", color="green" if ok else "red")
    plt.axis("off")
plt.suptitle("MobileNetV2 predictions on test images (red = wrong)")
plt.savefig(FIG_DIR / "test_predictions.png", bbox_inches="tight")
plt.show()

## 10. Export the model for the backend

In [ ]:
MODEL_FILE = "cats_dogs_mobilenetv2.keras"
model.save(OUT_DIR / MODEL_FILE)

info = {
    "model_file": MODEL_FILE,
    "img_size": IMG_SIZE,
    "class_names": CLASS_NAMES,
    "output": "single sigmoid unit = probability of class_names[1] (Dog)",
    "input": "RGB float32 in [0, 255], bilinear resize to img_size; normalization is inside the model",
    "test_accuracy": results["MobileNetV2"]["accuracy"],
    "baseline_test_accuracy": results["Baseline CNN"]["accuracy"],
    "train_minutes": {"baseline": round(baseline_time / 60, 1), "transfer": round(transfer_time / 60, 1)},
    "dataset_sizes": {"train": len(train_p), "val": len(val_p), "test": len(test_p)},
    "tensorflow_version": tf.__version__,
    "keras_version": keras.__version__,
    "trained_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
(OUT_DIR / "model_info.json").write_text(json.dumps(info, indent=2))
print(json.dumps(info, indent=2))

### Sanity check

Reload the saved file and predict a raw image file exactly the way the Flask backend will do it.

In [ ]:
reloaded = keras.models.load_model(OUT_DIR / MODEL_FILE)

for path, label in list(zip(test_p, test_y))[:5]:
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    p = float(reloaded.predict(img[tf.newaxis, ...], verbose=0)[0, 0])
    print(f"{path:30s} true={CLASS_NAMES[label]:3s}  P(dog)={p:.4f}")

In [ ]:
shutil.make_archive("cats_dogs_export", "zip", OUT_DIR)
print("Created cats_dogs_export.zip (model + model_info.json + figures)")

try:
    from google.colab import files
    files.download("cats_dogs_export.zip")
except ImportError:
    print("Not running in Colab — the zip is in the current folder.")